# Model Analysis — Test Set Accuracy

Evaluate one or more trained models against the held-out test split.  
Results can be saved back into each run's `training_history.json` for later grid-search comparison.

In [1]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

def find_project_root(marker_files=("requirements.txt", "README.md")):
    current = Path.cwd().resolve()
    for path in [current] + list(current.parents):
        if all((path / m).exists() for m in marker_files):
            return path
    raise FileNotFoundError("Could not find project root.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/marc/Documents/VS-Code Projects/captcha-ai


## Configuration

Edit `RUN_DIRS` to point at the experiment folders you want to evaluate.  
`DATA_DIR` and `CHARSET` are auto-discovered from `data/processed/` if left as `None`.

In [ ]:
# ── Configuration — edit this cell ───────────────────────────────────────────

# Paths to experiment folders (relative to project root, or absolute)
RUN_DIRS = [
    "experiments/succesful/test_run_20260401_1644",
    # "experiments/Debugging/test_run_20260401_2028",
]

# Dataset — set to None to auto-discover the first available processed dataset
DATA_DIR = None   # e.g. "data/processed/5Char_100k_Num_grayscale"
CHARSET  = None   # e.g. "0123456789"  or  "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

# Device override — set to None to use whatever is in each run's config.json
DEVICE = None     # e.g. "cpu", "mps", "cuda"

## Evaluate

In [ ]:
# ── Resolve dataset ──────────────────────────────────────────────────────────
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from get_accuracy import find_processed_dataset, evaluate_run

data_dir = 
charset  = CHARSET

if data_dir is None or charset is None:
    discovered_dir, discovered_charset = find_processed_dataset(PROJECT_ROOT)
    if discovered_dir is None:
        raise RuntimeError(
            "No processed dataset found under data/processed/. "
            "Set DATA_DIR and CHARSET manually in the config cell."
        )
    data_dir = data_dir or discovered_dir
    charset  = charset  or discovered_charset

print(f"Dataset : {data_dir}")
print(f"Charset : {charset!r}  ({len(charset)} classes)")

# ── Run evaluation ────────────────────────────────────────────────────────────
results = []
for run_dir in RUN_DIRS:
    # Accept both absolute paths and paths relative to project root
    path = Path(run_dir)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    print(f"\nEvaluating: {path.name} ...")
    try:
        result = evaluate_run(str(path), data_dir, charset, DEVICE)
        results.append(result)
        print(f"  Char Acc: {result['char_acc']*100:.2f}%  |  Seq Acc: {result['seq_acc']*100:.2f}%")
    except Exception as e:
        print(f"  [ERROR] {e}")

print(f"\nDone — {len(results)}/{len(RUN_DIRS)} run(s) evaluated.")

NameError: name 'DATA_DIR' is not defined

## Results Table

In [ ]:
import pandas as pd

if not results:
    print("No results to display.")
else:
    rows = []
    for r in results:
        row = {
            "run":       r["run"],
            "model":     r["model_name"],
            "device":    r["device"],
            "char_acc":  f"{r['char_acc']*100:.2f}%",
            "seq_acc":   f"{r['seq_acc']*100:.2f}%",
        }
        for i, acc in enumerate(r["pos_accs"]):
            row[f"pos_{i}"] = f"{acc*100:.1f}%"
        rows.append(row)

    df = pd.DataFrame(rows).set_index("run")
    display(df)

## Accuracy Chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not results:
    print("No results to plot.")
else:
    labels     = [r["run"] for r in results]
    char_accs  = [r["char_acc"] * 100 for r in results]
    seq_accs   = [r["seq_acc"]  * 100 for r in results]
    label_len  = len(results[0]["pos_accs"])
    pos_accs   = [[r["pos_accs"][i] * 100 for r in results] for i in range(label_len)]

    x      = np.arange(len(labels))
    n_bars = 2 + label_len
    width  = 0.8 / n_bars

    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 2.5), 5))

    ax.bar(x - (n_bars / 2 - 0.5) * width,       char_accs, width, label="Char Acc",  color="#4C72B0")
    ax.bar(x - (n_bars / 2 - 1.5) * width,        seq_accs,  width, label="Seq Acc",   color="#DD8452")
    for i in range(label_len):
        ax.bar(x - (n_bars / 2 - 2.5 - i) * width, pos_accs[i], width, label=f"Pos {i}", alpha=0.7)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Test Set Accuracy by Run")
    ax.set_ylim(0, 105)
    ax.legend(loc="lower right", fontsize=8)
    ax.axhline(100, color="grey", linewidth=0.5, linestyle="--")
    plt.tight_layout()
    plt.show()

## Save Results to `training_history.json`

Run this cell to write the test metrics back into each run's history file.  
This lets you compare runs in grid-search analysis later — look for `test_char_acc`, `test_seq_acc`, and `test_pos_acc_<i>` keys.

In [ ]:
import json

for r in results:
    # Locate the run folder
    run_path = Path(r["run"])
    if not run_path.is_absolute():
        run_path = PROJECT_ROOT / run_path
    # Fallback: run name is just the folder stem, reconstruct from RUN_DIRS
    if not run_path.exists():
        match = next(
            (PROJECT_ROOT / d for d in RUN_DIRS if Path(d).name == r["run"]),
            None,
        )
        if match:
            run_path = match

    history_path = run_path / "training_history.json"
    if not history_path.exists():
        print(f"[SKIP] {r['run']}: training_history.json not found")
        continue

    with open(history_path) as f:
        history = json.load(f)

    # Write test metrics (overwrites if already present)
    history["test_char_acc"] = r["char_acc"]
    history["test_seq_acc"]  = r["seq_acc"]
    for i, acc in enumerate(r["pos_accs"]):
        history[f"test_pos_acc_{i}"] = acc

    with open(history_path, "w") as f:
        json.dump(history, f, indent=4)

    print(f"[SAVED] {r['run']}  →  test_seq_acc={r['seq_acc']*100:.2f}%")